## Imports

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

In [2]:
np.random.seed(123)

## Data Generation

### 1. ERP

##### 1.1 ERP: Products

In [3]:
products_data = [
    ["FB_V1","FRONT_BUMPER","CUST_A",55],
    ["FB_V2","FRONT_BUMPER","CUST_B",60],
    ["RB_V1","REAR_BUMPER","CUST_A",45],
    ["RB_V2","REAR_BUMPER","CUST_B",50],
    ["DB_V1","DASHBOARD","CUST_A",75],
    ["DB_V2","DASHBOARD"," CUST_C",80],
    ["DP_V1","DOOR_PANEL","CUST_A",45],
    ["DP_V2","DOOR_PANEL"," CUST_C",50],
]

In [4]:
df_products = pd.DataFrame(products_data, columns=["product_code","product_base","customer_specific","standard_cycle_time_sec"])

In [5]:
df_products.head()

,product_code,product_base,customer_specific,standard_cycle_time_sec
0,FB_V1,FRONT_BUMPER,CUST_A,55
1,FB_V2,FRONT_BUMPER,CUST_B,60
2,RB_V1,REAR_BUMPER,CUST_A,45
3,RB_V2,REAR_BUMPER,CUST_B,50
4,DB_V1,DASHBOARD,CUST_A,75


##### 1.2 ERP: BOM

In [6]:
bom_data = []
for product in df_products["product_code"]:
    base = df_products.loc[df_products.product_code==product, "product_base"].values[0]
    
    if base in ["FRONT_BUMPER","REAR_BUMPER"]:
        bom_data.append([product,"MAT_PP_EPDM",round(np.random.normal(3.5,0.1),3),"kg"])
        bom_data.append([product,"INSERT_BRACKET",2,"pcs"])
        bom_data.append([product,"PAINT_BASE",round(np.random.normal(0.35,0.02),3),"kg"])
        bom_data.append([product,"PAINT_CLEAR",round(np.random.normal(0.2,0.01),3),"kg"])
        bom_data.append([product,"HARDENER",round(np.random.normal(0.02,0.005),3),"kg"])
        bom_data.append([product,"SOLVENT",round(np.random.normal(0.05,0.01),3),"kg"])
        bom_data.append([product,"PACK_BOX_L",1,"pcs"])
    elif base=="DASHBOARD":
        bom_data.append([product,"MAT_PC_ABS",round(np.random.normal(2.0,0.1),3),"kg"])
        bom_data.append([product,"FOAM_LAYER",round(np.random.normal(0.4,0.01),3),"kg"])
        bom_data.append([product,"INSERT_BRACKET",1,"pcs"])
        bom_data.append([product,"SOFT_TOUCH_COATING",round(np.random.normal(0.1,0.01),3),"kg"])
        bom_data.append([product,"PACK_BOX_M",1,"pcs"])
    elif base=="DOOR_PANEL":
        bom_data.append([product,"MAT_PP_ABS",round(np.random.normal(2.5,0.05),3),"kg"])
        bom_data.append([product,"INSERT_BRACKET",1,"pcs"])
        bom_data.append([product,"FOAM_LAYER",round(np.random.normal(0.3,0.02),3),"kg"])
        bom_data.append([product,"PACK_BOX_m",1,"pcs"])

In [7]:
df_bom = pd.DataFrame(bom_data, columns=["product_code","component_code","component_qty","unit"])

In [8]:
df_bom.head()

,product_code,component_code,component_qty,unit
0,FB_V1,MAT_PP_EPDM,3.391,kg
1,FB_V1,INSERT_BRACKET,2.000,pcs
2,FB_V1,PAINT_BASE,0.370,kg
3,FB_V1,PAINT_CLEAR,0.203,kg
4,FB_V1,HARDENER,0.012,kg


##### 1.3 ERP: Customers

In [9]:
customers_data = [
    ["CUST_A", " Future Transport Technologies ", "1900-01-01", "2026-12-31"],
    ["CUST_B", "Mobility Solutions", "2025-02-01", "2027-03-31"],
    [" CUST_C", " Vehicle Engineering Systems", "2018-01-01", "2026-12-31"],
    ["CUST_D", "Driving Intelligence ", "2019-01-01", "2025-06-31"],
    ["CUST_D", "Driving Intelligence ", "2019-01-01", "2024-06-31"]
]

In [10]:
df_customers = pd.DataFrame(customers_data, columns=["customer_code","customer_name","valid_from","valid_to"])

In [11]:
df_customers.head()

,customer_code,customer_name,valid_from,valid_to
0,CUST_A,Future Transport Technologies,1900-01-01,2026-12-31
1,CUST_B,Mobility Solutions,2025-02-01,2027-03-31
2,CUST_C,Vehicle Engineering Systems,2018-01-01,2026-12-31
3,CUST_D,Driving Intelligence,2019-01-01,2025-06-31
4,CUST_D,Driving Intelligence,2019-01-01,2024-06-31


### 2. MES

##### 2.1 MES: Production Orders

In [12]:
line_mapping = {
    "FB_V1":"LINE_1","FB_V2":"LINE_1",
    "RB_V1":"L2","RB_V2":"LINE_2",
    "DB_V1":"LINE_3","DB_V2":"LINE_3",
    "DP_V1":"L4","DP_V2":"LINE_4"
}

line_current_time = {line: datetime(2026,1,1,6,0) for line in line_mapping.values()}

total_records = 56256
products_list = df_products["product_code"].tolist()
weights = [0.15,0.15,0.15,0.15,0.1,0.1,0.1,0.1]

selected_products = np.random.choice(products_list, size=total_records, p=weights)

mes_data = []
mes_id = 1

for product in selected_products:
    line = line_mapping[product]
    current_time = line_current_time[line]
    
    # Shift Day/Night
    if current_time.hour < 14:
        shift = "Day"
        shift_end = current_time.replace(hour=14, minute=0)
    else:
        shift = "2"
        shift_end = current_time.replace(hour=22, minute=0)
    
    # Setup/Shift change
    if current_time.minute==0 and current_time.second==0:
        current_time += timedelta(minutes=15)
    
    planned = np.random.randint(30,50)
    produced = planned - np.random.randint(0,10)
    produced_pass = produced - np.random.randint(0,5)
    scrap = produced - produced_pass
    
    cycle = df_products.loc[df_products.product_code==product, "standard_cycle_time_sec"].values[0]
    cycle_actual = round(np.random.normal(cycle,5),1)
    
    duration_minutes = (planned * cycle_actual) / 60
    inter_job_break = np.random.randint(5,16)
    
    start_time = current_time + timedelta(minutes=inter_job_break)
    end_time = start_time + timedelta(minutes=duration_minutes)
    line_current_time[line] = end_time
    
    customer_code = df_products.loc[df_products.product_code==product,"customer_specific"].values[0]
    
    mes_data.append([
        mes_id,
        f"PO_{1000+mes_id}",
        product,
        customer_code,
        line,
        start_time,
        end_time,
        shift,
        planned,
        produced,
        produced_pass,
        scrap,
        cycle_actual
    ])
    mes_id += 1

In [13]:
df_mes = pd.DataFrame(mes_data, columns=[
    "mes_id","production_order","product_code","customer_code",
    "production_line","start_time","end_time","shift",
    "planned_qty","produced_qty","produced_pass_qty", "scrap_qty","cycle_time_sec"
])

In [14]:
df_mes.head()

,mes_id,production_order,product_code,customer_code,production_line,start_time,end_time,shift,planned_qty,produced_qty,produced_pass_qty,scrap_qty,cycle_time_sec
0,1,PO_1001,RB_V1,CUST_A,L2,2026-01-01 06:20:00.000,2026-01-01 06:53:57.200,Day,44,44,42,2,46.3
1,2,PO_1002,RB_V2,CUST_B,LINE_2,2026-01-01 06:23:00.000,2026-01-01 07:06:32.800,Day,46,44,43,1,56.8
2,3,PO_1003,RB_V1,CUST_A,L2,2026-01-01 06:59:57.200,2026-01-01 07:29:57.200,Day,45,38,34,4,40.0
3,4,PO_1004,RB_V1,CUST_A,L2,2026-01-01 07:44:57.200,2026-01-01 08:11:14.200,Day,38,32,30,2,41.5
4,5,PO_1005,RB_V1,CUST_A,L2,2026-01-01 08:25:14.200,2026-01-01 08:56:54.800,Day,43,36,34,2,44.2


##### 2.2 MES: Production plants

In [15]:
plants_data = [
    ["PLANT_1", "Plant 1", "Brzechin", "Poland", "Europe"],
    ["PLANT_2", "Plant 2", "Lindenholt", "Germany", "Europe"],
    ["PLANT_3", "Plant 3", "Lipovník", "Czech Republic", "Europe"],
    ["PLANT_4", "Plant 4", "Saint-Aubrel", "France", "Europe"],
]

In [16]:
df_mes_plants = pd.DataFrame(plants_data, columns=[
        "plant_code",
        "plant_name",
        "city",
        "country",
        "region"
])

In [17]:
df_mes_plants.head()

,plant_code,plant_name,city,country,region
0,PLANT_1,Plant 1,Brzechin,Poland,Europe
1,PLANT_2,Plant 2,Lindenholt,Germany,Europe
2,PLANT_3,Plant 3,Lipovník,Czech Republic,Europe
3,PLANT_4,Plant 4,Saint-Aubrel,France,Europe


##### 2.3 MES: Production lines

In [18]:
lines_data = [
    ["LINE_1", "Front Bumper Line", "PLANT_1", "Automotive Exterior"],
    ["LINE_2", "Rear Bumper Line", "PLANT_1", "Automotive Exterior"],
    ["LINE_3", "Dashboard Line", "PLANT_1", "Automotive Interior"],
    ["LINE_4", "Door Panel Line", "PLANT_2", "Automotive Interior"]
]

In [19]:
df_mes_lines = pd.DataFrame(lines_data, columns=[
        "line_code",
        "line_name",
        "plant",
        "production_area"
])

In [20]:
df_mes_lines.head()

,line_code,line_name,plant,production_area
0,LINE_1,Front Bumper Line,PLANT_1,Automotive Exterior
1,LINE_2,Rear Bumper Line,PLANT_1,Automotive Exterior
2,LINE_3,Dashboard Line,PLANT_1,Automotive Interior
3,LINE_4,Door Panel Line,PLANT_2,Automotive Interior


##### 2.4 MES: Machines

In [21]:
machines_data = [
    # Front Bumper
    ["INJ_01", "Injection FB Machine 1", "LINE_1", "INJ", True],
    ["PNT_01", "Paint Booth FB 1", "LINE_1", "PNT", True],

    # Rear Bumper
    ["INJ_01", "Injection RB Machine 1", "LINE_2", "INJ", True],
    ["PNT_01", "Paint Booth RB 1", "LINE_2", "PNT", True],

    # Dashboard
    ["INJ_01", "Injection Dashboard 1", "LINE_3", "INJ", True],
    ["STC_01", "Soft Touch DB 1", "LINE_3", "STC", True],

    # Door Panel
    ["INJ_01", "Injection Door Panel 1", "LINE_4", "INJ", True]
]

In [22]:
df_mes_machines = pd.DataFrame(machines_data, columns=[
        "machine_code",
        "machine_name",
        "line_code",
        "process_code",
        "is_active"
])

In [23]:
df_mes_machines.head()

,machine_code,machine_name,line_code,process_code,is_active
0,INJ_01,Injection FB Machine 1,LINE_1,INJ,True
1,PNT_01,Paint Booth FB 1,LINE_1,PNT,True
2,INJ_01,Injection RB Machine 1,LINE_2,INJ,True
3,PNT_01,Paint Booth RB 1,LINE_2,PNT,True
4,INJ_01,Injection Dashboard 1,LINE_3,INJ,True


##### 2.5 MES: Processes

In [24]:
process_data = [
    ["INJ", "Injection Molding", "Molding", "Plastic Processing"],
    ["PNT", "Painting", "Paint Shop", "Surface Treatment"],
    ["STC", "Soft Touch Coating", "Coating", "Surface Treatment"]
]

In [25]:
df_mes_processes = pd.DataFrame(process_data, columns=[
        "process_code",
        "process_name",
        "department",
        "technology_group"
])

In [26]:
df_mes_processes.head()

,process_code,process_name,department,technology_group
0,INJ,Injection Molding,Molding,Plastic Processing
1,PNT,Painting,Paint Shop,Surface Treatment
2,STC,Soft Touch Coating,Coating,Surface Treatment


##### 2.6 MES: Shifts

In [27]:
shifts_data = [
    ["DAY_1", "Day Shift", "06:00:00", "14:00:00", 8],
    ["NIGHT_2", "Night Shift", "14:00:00", "22:00:00", 8]
]

In [28]:
df_mes_shifts = pd.DataFrame(shifts_data, columns=[
        "shift_code",
        "shift_name",
        "shift_start_time",
        "shift_end_time",
        "shift_duration_hours"
])

In [29]:
df_mes_shifts.head()

,shift_code,shift_name,shift_start_time,shift_end_time,shift_duration_hours
0,DAY_1,Day Shift,06:00:00,14:00:00,8
1,NIGHT_2,Night Shift,14:00:00,22:00:00,8


### 3. IOT

##### 3.1 IOT: Process Data

In [30]:
iot_rows = []
iot_id = 1

for _, row in df_mes.iterrows():
    start = row["start_time"]
    end = row["end_time"]
    product_base = df_products.loc[df_products.product_code==row["product_code"], "product_base"].values[0]
    
    timestamps = [start + (end-start)*i/2 for i in range(3)]
    
    for ts in timestamps:
        temperature = round(np.random.normal(220,10),1)
        pressure = round(np.random.normal(1200,150),0)
        humidity = round(np.random.uniform(30,60),1)
        flow_rate = round(np.random.uniform(20,50),1)

        stage = "INT"

        # Painting
        if product_base in ["FRONT_BUMPER","REAR_BUMPER"]:
            temperature = round(np.random.normal(23,2),1)
            pressure = round(np.random.normal(2,0.3),2)
            humidity = round(np.random.uniform(40,70),1)
            flow_rate = round(np.random.uniform(0.2,0.8),2)
            stage = "PNT"

        # Soft Touch Coating
        elif product_base == "DASHBOARD":
            temperature = round(np.random.normal(25,2),1)
            pressure = round(np.random.normal(2.5,0.4),2)
            humidity = round(np.random.uniform(45,65),1)
            flow_rate = round(np.random.uniform(0.3,0.7),2)
            stage = "STC"
        
        iot_rows.append([
            iot_id,
            row["mes_id"],
            ts,
            temperature,
            pressure,
            humidity,
            flow_rate,
            stage
        ])
        iot_id += 1

In [31]:
df_iot = pd.DataFrame(iot_rows, columns=[
    "iot_id","mes_id","date_hour",
    "temperature_c","pressure_bar","humidity_pct","flow_rate_lpm", "stage"
])

In [32]:
df_iot.head()

,iot_id,mes_id,date_hour,temperature_c,pressure_bar,humidity_pct,flow_rate_lpm,stage
0,1,1,2026-01-01 06:20:00.000,19.5,1.60,43.3,0.55,PNT
1,2,1,2026-01-01 06:36:58.600,18.5,2.32,46.1,0.54,PNT
2,3,1,2026-01-01 06:53:57.200,18.9,2.25,43.9,0.61,PNT
3,4,2,2026-01-01 06:23:00.000,24.6,1.87,68.7,0.74,PNT
4,5,2,2026-01-01 06:44:46.400,23.6,1.77,41.7,0.79,PNT


### 4. QMS

##### 4.1 QMS: Quality Inspections 

In [33]:
qms_rows = []
qms_id = 1

product_defects = {
    "FRONT_BUMPER":["bubble","thin_coating","adhesion_failure","orange_peel","short_shot","sink-mark"],
    "REAR_BUMPER":["bubble","orange_peel","adhesion_failure","thin_coating","short_shot","sink-mark"],
    "DASHBOARD":["delamination","sink-mark","color_variation","short_shot","adhesion_failure","bubble"],
    "DOOR_PANEL":["short_shot","flash","warping"]
}

for _, row in df_mes.iterrows():
    product_base = df_products.loc[df_products.product_code==row["product_code"], "product_base"].values[0]
    defects_list = product_defects[product_base]
    
    has_defect = np.random.rand() < 0.10  # ~10% to hold
    
    if has_defect:
        num_defects = np.random.randint(1,len(defects_list)+1)
        sampled_defects = np.random.choice(len(defects_list), num_defects, replace=False)
        for i in sampled_defects:
            defect_type = defects_list[i]
            defect_qty = np.random.randint(1,4)
            inspection_time = row["start_time"] + timedelta(
                seconds = np.random.randint(0,int((row["end_time"]-row["start_time"]).total_seconds()))
            )
            qms_rows.append([
                qms_id,
                row["production_order"],
                row["product_code"],
                row["customer_code"],
                inspection_time,
                row["shift"],
                defect_type,
                defect_qty
            ])
            qms_id += 1
    else:
        inspection_time = row["end_time"]
        qms_rows.append([
            qms_id,
            row["production_order"],
            row["product_code"],
            row["customer_code"],
            inspection_time,
            row["shift"],
            None,
            0
        ])
        qms_id += 1


In [34]:
df_qms = pd.DataFrame(qms_rows, columns=[
    "qms_id","production_order","product_code","customer_code",
    "inspection_time","shift","defect_type","defect_qty"
])

In [53]:
df_qms.head()

,qms_id,production_order,product_code,customer_code,inspection_time,shift,defect_type,defect_qty
0,1,PO_1001,RB_V1,CUST_A,2026-01-01 06:30:34,Day,short_shot,2
1,2,PO_1001,RB_V1,CUST_A,2026-01-01 06:38:17,Day,adhesion_failure,3
2,3,PO_1001,RB_V1,CUST_A,2026-01-01 06:28:57,Day,thin_coating,1
3,4,PO_1001,RB_V1,CUST_A,2026-01-01 06:23:56,Day,orange_peel,3
4,5,PO_1001,RB_V1,CUST_A,2026-01-01 06:38:20,Day,sink-mark,1


In [36]:
df_qms[df_qms['defect_qty']> 0]

,qms_id,production_order,product_code,customer_code,inspection_time,shift,defect_type,defect_qty
0,1,PO_1001,RB_V1,CUST_A,2026-01-01 06:30:34.000,Day,short_shot,2
1,2,PO_1001,RB_V1,CUST_A,2026-01-01 06:38:17.000,Day,adhesion_failure,3
2,3,PO_1001,RB_V1,CUST_A,2026-01-01 06:28:57.000,Day,thin_coating,1
3,4,PO_1001,RB_V1,CUST_A,2026-01-01 06:23:56.000,Day,orange_peel,3
4,5,PO_1001,RB_V1,CUST_A,2026-01-01 06:38:20.000,Day,sink-mark,1
...,...,...,...,...,...,...,...,...
68293,68294,PO_57238,RB_V1,CUST_A,2026-08-21 13:09:11.200,Day,short_shot,2
68294,68295,PO_57239,FB_V1,CUST_A,2027-07-14 06:31:39.500,Day,thin_coating,3
68295,68296,PO_57239,FB_V1,CUST_A,2027-07-14 06:15:26.500,Day,sink-mark,2
68296,68297,PO_57239,FB_V1,CUST_A,2027-07-14 06:35:11.500,Day,bubble,2


##### 4.2 QMS: Defect Catalog

In [37]:
defects_dict = {
    "PNT":[("thin_coating","major", "P1"),("adhesion_failure","major", "P2"),
           ("orange_peel","minor", "P3"),("dust_inclusion","minor", "P4")],
    "STC":[("delamination","major", "S1"),("color_variation","minor", "S2"),
           ("adhesion_failure","major", "S3"),("uneven_surface","minor", "S4")],
    "INT":[("short_shot","major", "I1"),("flash","minor", "I2"),("warping","major","I3"),
           ("sink_mark","minor","I4"),("bubble","minor","I5")]
}

rows = []

for stage, defects in defects_dict.items():
    for defect_type, defect_category, defect_code in defects:
        rows.append([defect_type, defect_category, defect_code, stage])

df_qms_defect_catalog = pd.DataFrame(rows, columns=["defect_type","defect_category","defect_code","stage"])

In [38]:
df_qms_defect_catalog.head()

,defect_type,defect_category,defect_code,stage
0,thin_coating,major,P1,PNT
1,adhesion_failure,major,P2,PNT
2,orange_peel,minor,P3,PNT
3,dust_inclusion,minor,P4,PNT
4,delamination,major,S1,STC


## CSV Export

In [39]:
raw_path_erp = Path("../data/source_erp")
raw_path_erp.mkdir(parents=True, exist_ok=True)

raw_path_mes = Path("../data/source_mes")
raw_path_mes.mkdir(parents=True, exist_ok=True)

raw_path_iot = Path("../data/source_iot")
raw_path_iot.mkdir(parents=True, exist_ok=True)

raw_path_qms = Path("../data/source_qms")
raw_path_qms.mkdir(parents=True, exist_ok=True)

In [40]:
df_products.to_csv(
    raw_path_erp/"erp_products.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [41]:
df_bom.to_csv(
    raw_path_erp/"erp_bom.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [42]:
df_customers.to_csv(
    raw_path_erp/"erp_customers.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [43]:
df_mes.to_csv(
    raw_path_mes/"mes_production_orders.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [44]:
df_mes_plants.to_csv(
    raw_path_mes/"mes_plants.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [45]:
df_mes_lines.to_csv(
    raw_path_mes/"mes_lines.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [46]:
df_mes_machines.to_csv(
    raw_path_mes/"mes_machines.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [47]:
df_mes_processes.to_csv(
    raw_path_mes/"mes_processes.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [48]:
df_mes_shifts.to_csv(
    raw_path_mes/"mes_shifts.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [49]:
df_iot.to_csv(
    raw_path_iot/"iot_process_data.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [50]:
df_qms.to_csv(
    raw_path_qms/"qms_inspections.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

In [51]:
df_qms_defect_catalog.to_csv(
    raw_path_qms/"qms_defect_catalog.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)